# Kriging (nugget mode) on the noisy Branin 2D function (Octave)

This notebook demonstrates `Kriging` with `noise="nugget"`, which estimates a **global nugget**
effect (homoscedastic noise variance) from the data. Unlike `Kriging` with `noise` vector
where noise variances are known and provided per point, here the nugget is
a single scalar that is optimised alongside the other hyperparameters.

Steps:
1. Setup mlibkriging
2. Define the Branin function and add noise
3. Build a space-filling design and evaluate it
4. Fit a `Kriging` with `noise="nugget"` model
5. Predict on a fine grid and plot mean + uncertainty
6. Inspect model parameters (including the estimated nugget)

## 0. Setup

Build the C++ core and the Octave binding from source (skip if already built).
Requires: `cmake`, a C++ compiler, Octave ≥ 6.0.

In [1]:
% Add mlibkriging to path
% Adjust the path below to your build/installed directory
repo_root = fullfile(fileparts(pwd()), '..');
build_path = fullfile(repo_root, 'build', 'installed', 'bindings', 'Octave');
if ~exist(fullfile(build_path, 'mLibKriging.mex'), 'file') ...
   && ~exist(fullfile(build_path, ['mLibKriging.', mexext]), 'file')
    error('mlibkriging not found at %s — please build first (see README.md)', build_path);
end
addpath(build_path);
addpath(fullfile(repo_root, 'bindings', 'Octave', 'mlibkriging'));
disp('mlibkriging loaded')

mlibkriging loaded


## 1. Noisy Branin function

Same as the `Kriging` with `noise` vector example: the Branin function on $[0,1]^2$
with additive Gaussian noise ($\sigma_{\varepsilon} = 5$).

The difference is that here we **do not tell** the model the noise level.
Instead, `Kriging` with `noise="nugget"` will estimate a single global nugget from the data.

In [2]:
function z = branin(X)
    x1 = X(:,1) * 15 - 5;
    x2 = X(:,2) * 15;
    z = (x2 - 5/(4*pi^2) * x1.^2 + 5/pi * x1 - 6).^2 ...
        + 10 * (1 - 1/(8*pi)) * cos(x1) + 10;
end

% 50x50 evaluation grid
grid_x = linspace(0, 1, 50);
[G1, G2] = meshgrid(grid_x, grid_x);
grid_pts = [G1(:), G2(:)];
z_true = reshape(branin(grid_pts), 50, 50);

figure;
contourf(G1, G2, z_true, 20);
colorbar;
title('True Branin function');
xlabel('x_1'); ylabel('x_2');

Inline plot failed, consider trying another graphics toolkit
error: print: directory /tmp/tmpgq4f3y_t/plots/tmpo4pzycm0 does not exist
error: called from
    _make_figures>safe_print at line 125 column 7
    _make_figures at line 49 column 13



## 2. Design of experiments (with noise)

We sample $n = 40$ points using a Latin Hypercube Design and add
Gaussian noise with $\sigma_{\varepsilon} = 5$.

Unlike `Kriging` with `noise` vector, we do **not** pass noise variances — the model
estimates a single nugget parameter.

In [3]:
rand('seed', 42); randn('seed', 42);
n = 40; d = 2;
noise_sd = 5.0;

% Simple LHS
X = zeros(n, d);
for j = 1:d
    perm = randperm(n);
    X(:,j) = (perm' - rand(n,1)) / n;
end
y_true = branin(X);
y = y_true + noise_sd * randn(n, 1);

figure;
contourf(G1, G2, z_true, 20);
hold on;
scatter(X(:,1), X(:,2), 60, 'w', 'filled', 'MarkerEdgeColor', 'k');
hold off;
colorbar;
title(sprintf('%d LHS points (noise sigma = %.1f)', n, noise_sd));
xlabel('x_1'); ylabel('x_2');

Inline plot failed, consider trying another graphics toolkit
error: print: directory /tmp/tmpgq4f3y_t/plots/tmpu5166mp7 does not exist
error: called from
    _make_figures>safe_print at line 125 column 7
    _make_figures at line 49 column 13



## 3. Fit a Kriging (nugget mode) model

`Kriging(noise="nugget")` has the same interface as `Kriging()` (no `noise` argument).
It automatically estimates the nugget effect during fitting.

In [4]:
k = Kriging(y, X, 'matern5_2', 'constant', false, 'BFGS10', 'LL', [], 'nugget');
disp(k.summary())

* data: 40x[0.00416876,0.998323],[0.017024,0.977671] -> 40x[1.03251,170.872]
* trend constant (est.): 174.254
* variance (est.): 45422.2
* covariance:
  * kernel: matern5_2
  * range (est.): 0.631985, 1.5098
  * nugget (est.): 14.6804
  * fit:
    * objective: LL
    * optim: BFGS10



## 4. Predict on a fine grid

`predict()` returns the posterior mean and standard deviation.
The mean is a smooth estimate of the underlying function.

In [5]:
[p_mean, p_stdev] = k.predict(grid_pts, true, false, false);
z_mean = reshape(p_mean, 50, 50);
z_sd   = reshape(p_stdev, 50, 50);

vmin = min(min(z_true(:)), min(z_mean(:)));
vmax = max(max(z_true(:)), max(z_mean(:)));

figure;
subplot(1, 2, 1);
contourf(G1, G2, z_true, 20);
hold on;
scatter(X(:,1), X(:,2), 40, 'w', 'filled', 'MarkerEdgeColor', 'k');
hold off;
caxis([vmin vmax]); colorbar;
title('True Branin'); xlabel('x_1'); ylabel('x_2');

subplot(1, 2, 2);
contourf(G1, G2, z_mean, 20);
hold on;
scatter(X(:,1), X(:,2), 40, 'w', 'filled', 'MarkerEdgeColor', 'k');
hold off;
caxis([vmin vmax]); colorbar;
title('Kriging (nugget mode) mean'); xlabel('x_1'); ylabel('x_2');

Inline plot failed, consider trying another graphics toolkit
error: print: directory /tmp/tmpgq4f3y_t/plots/tmpw4e8x761 does not exist
error: called from
    _make_figures>safe_print at line 125 column 7
    _make_figures at line 49 column 13



In [6]:
% Posterior standard deviation (uncertainty)
figure;
contourf(G1, G2, z_sd, 20);
hold on;
scatter(X(:,1), X(:,2), 60, 'w', 'filled', 'MarkerEdgeColor', 'k');
hold off;
colorbar;
title('Kriging (nugget mode) std dev (uncertainty)');
xlabel('x_1'); ylabel('x_2');

Inline plot failed, consider trying another graphics toolkit
error: print: directory /tmp/tmpgq4f3y_t/plots/tmprxqgh3px does not exist
error: called from
    _make_figures>safe_print at line 125 column 7
    _make_figures at line 49 column 13



## 5. Model inspection

Key fitted parameters: length-scales $\theta$, variance $\sigma^2$,
**nugget**, and the log-likelihood at the optimum.

Compare the estimated noise variance (nugget) with the true
$\sigma_{\varepsilon}^2 = 25$.

In [7]:
fprintf('Kernel       : %s\n', k.kernel());
fprintf('Theta (range): %s\n', mat2str(k.theta(), 4));
fprintf('Sigma2       : %.4f\n', k.sigma2());
fprintf('Nugget       : %.4f\n', k.nugget());
fprintf('LogLikelihood: %.4f\n', k.logLikelihood());
fprintf('\nTrue noise variance:      %.2f\n', noise_sd^2);

Kernel       : matern5_2


Theta (range): [0.632;1.51]


Sigma2       : 45422.2255


Nugget       : 14.6804


LogLikelihood: -156.9939



True noise variance:      25.00


In [8]:
% Log-likelihood surface over (theta1, theta2) at estimated alpha
% Kriging (nugget mode)::logLikelihoodFun expects [theta1; theta2; alpha]
% with alpha = sigma2 / (sigma2 + nugget)
alpha_hat = k.sigma2() / (k.sigma2() + k.nugget());
theta_vals = linspace(0.05, 3.0, 40);
[T1, T2] = meshgrid(theta_vals, theta_vals);
ll_mat = zeros(40, 40);
for i = 1:40
    for j = 1:40
        ll_mat(i,j) = k.logLikelihoodFun([T1(i,j); T2(i,j); alpha_hat], false);
    end
end

figure;
contourf(T1, T2, ll_mat, 20);
colorbar;
hold on;
th = k.theta();
plot(th(1), th(2), 'r+', 'MarkerSize', 12, 'LineWidth', 2);
hold off;
title('Log-likelihood surface (at estimated alpha)');
xlabel('\theta_1'); ylabel('\theta_2');
legend('', 'optimum');

Inline plot failed, consider trying another graphics toolkit
error: print: directory /tmp/tmpgq4f3y_t/plots/tmp3nueohgb does not exist
error: called from
    _make_figures>safe_print at line 125 column 7
    _make_figures at line 49 column 13

